# SHAPE Evaluation

This evaluation assesses the effectiveness of SHAPE in transforming narrative clinical pathways into interoperable
FHIR PlanDefinition resources.

In [83]:
import json

with open("dataset/clinical_pathways.json", 'r') as f:
    clinical_pathways = json.load(f)

plan_names = [cp["name"] for cp in clinical_pathways]

In [84]:
from fhir.resources.plandefinition import PlanDefinition

plan_definitions_groundtruth = {}

for pname in plan_names:
    
    with open(f"dataset/groundtruth/{pname}.json", "r") as f:
        plandef = json.load(f)
        plan_definitions_groundtruth[pname] = PlanDefinition.model_validate(plandef)

In [85]:
from fhir.resources import condition
from os import path
from src.model.core import Relationship, ActionCondition
from src.model.enums import ParameterCategory, TemporalType
from typing import Any, Dict, List, Optional

class GroundTruthTemporalConstraint:
    label : str
    type : TemporalType
    resource : Dict[str, Any]

    def __init__(self, label : str, type : TemporalType, resource : Dict[str, Any]):
        self.label = label
        self.type = type
        self.resource = resource
    
    @staticmethod
    def from_dict(tc : Dict[str, Any]) -> "GroundTruthTemporalConstraint":
        return GroundTruthTemporalConstraint(
            label = tc['label'],
            type = TemporalType(tc['type']),
            resource = tc['resource'],
        )
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "label" : self.label,
            "type" : self.type.value,
            "resource" : self.resource,
        }

class GroundTruthParameter:
    label : str
    path : str
    category : ParameterCategory
    value : Optional[Any]
    possible_values : Optional[List[Any]]

    def __init__(self, label : str, path : str, category : ParameterCategory, value : Optional[Any] = None, possible_values : Optional[List[Any]] = None):
        self.label = label
        self.path = path
        self.category = category
        self.value = value
        self.possible_values = possible_values

    @staticmethod
    def from_dict(param : Dict[str, Any]) -> "GroundTruthParameter":
        return GroundTruthParameter(
            label = param["label"],
            path = param["path"],
            category = ParameterCategory(param["category"]),
            value = param.get("value"),
            possible_values = param.get("possible_values"),
        )
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "label": self.label,
            "path": self.path,
            "category": self.category.value,
            "value": self.value,
            "possible_values": self.possible_values,
        }

class GroundTruthAction:
    id : str
    label : str
    condition : Optional[ActionCondition]
    temporal_constraint : Optional[GroundTruthTemporalConstraint]
    activity_definition : str
    parameters : Optional[Dict[str, GroundTruthParameter]]

    def __init__(self, id : str, label : str, activity_definition : str, condition : Optional[ActionCondition] = None, temporal_constraint : Optional[GroundTruthTemporalConstraint] = None, parameters : Optional[Dict[str, GroundTruthParameter]] = None):
        self.id = id
        self.label = label
        self.condition = condition
        self.temporal_constraint = temporal_constraint
        self.activity_definition = activity_definition
        self.parameters = parameters
    
    @staticmethod
    def from_dict(action : Dict[str, Any]) -> "GroundTruthAction":
        gt_params = {}
        if action.get("parameters"):
            for param in action.get("parameters"):
                gt_param = GroundTruthParameter.from_dict(param)
                gt_params[gt_param.path] = gt_param
        return GroundTruthAction(
            id = action["id"],
            label = action["label"],
            condition = ActionCondition.from_dict(action.get("condition")) if action.get("condition") else None,
            temporal_constraint = GroundTruthTemporalConstraint.from_dict(action.get("temporal_constraint")) if action.get("temporal_constraint") else None,
            activity_definition = action["activity_definition"],
            parameters = gt_params if len(gt_params) > 0 else None,
        )
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "id": self.id,
            "label": self.label,
            "condition": self.condition.to_dict() if self.condition else None,
            "temporal_constraint": self.temporal_constraint.to_dict() if self.temporal_constraint else None,
            "activity_definition": self.activity_definition,
            "parameters": {key: value.to_dict() for key, value in self.parameters.items()} if self.parameters else None,
        }

class GroundTruthWorkflow:
    actions : Dict[str, GroundTruthAction]
    relationships : List[Relationship]

    def __init__(self, actions : Dict[str, GroundTruthAction], relationships : List[Relationship]):
        self.actions = actions
        self.relationships = relationships
    
    @staticmethod
    def from_dict(workflow: Dict[str, Any]) -> "GroundTruthWorkflow":
        return GroundTruthWorkflow(
            actions= {id: GroundTruthAction.from_dict(action) for id, action in workflow['actions'].items()},
            relationships=[Relationship.from_dict(rel) for rel in workflow['relationships']]
        )
    
    def to_dict(self):
        return {
            "actions" : {id : action.to_dict() for id, action in self.actions.items()},
            "relationships" : [rel.to_dict() for rel in self.relationships]
        }

In [86]:
with open(f"dataset/groundtruth/all_elements.json", "r") as f:
    all_elements_dict = json.load(f)

groundtruth_elements = {}

for pname, elements in all_elements_dict.items():
    temp_elements = {
        "actions" : {},
        "relationships" : elements["relationships"]
    }
    for act in elements['actions']:
        temp_elements["actions"][act['id']] = act
    groundtruth_elements[pname] = GroundTruthWorkflow.from_dict(temp_elements)

In [87]:
from src.sources.llm import llm_models
from src.model.fhir_workflow import FHIRWorkflow

generated_outputs = {}
for llm in llm_models.keys():
    generated_outputs[llm] = {}
    for pname in plan_names:
        with open(f"output/{llm}/{pname}_output.json", "r") as f:
            generated_output_dict = json.load(f)
            generated_outputs[llm][pname] = FHIRWorkflow.from_dict(generated_output_dict)

In [88]:
def compute_precision(tp, fp):
    return tp / (tp + fp) if tp + fp else 0

def compute_recall(tp, fn):
    return tp / (tp + fn) if tp + fn else 0

def compute_f1_score(precision_val, recall_val):
    return (
        2 * precision_val * recall_val / (precision_val + recall_val)
        if precision_val + recall_val else 0
    )

## RQ1 - Can SHAPE accurately reconstruct the logical structure of a narrative clinical pathway?

This research question evaluates the structural extraction of the pipeline. In particular, we assess SHAPE’s ability to
identify clinical actions, their execution conditions and temporal constraints, and reconstruct the workflow connecting
them through execution relationships.

In [89]:
import os

os.makedirs('evaluation/rq1', exist_ok=True)

In [90]:
generated_structural_elements= {}

for llm, plan_outputs in generated_outputs.items():
    generated_structural_elements[llm] = {}
    for pname, output in plan_outputs.items():
        generated_structural_elements[llm][pname] = {}
        
        generated_structural_elements[llm][pname]["actions"] = {}
        generated_structural_elements[llm][pname]["temporal_constraints"] = {}
        generated_structural_elements[llm][pname]["conditions"] = {}
        
        for fhir_action in output.fhir_actions:
            ## ACTIONS
            generated_structural_elements[llm][pname]["actions"][fhir_action.id] = fhir_action.label
            ## TEMPORAL CONSTRAINTS
            if fhir_action.temporal_constraint:
                generated_structural_elements[llm][pname]["temporal_constraints"][fhir_action.id] = fhir_action.temporal_constraint
            ## CONDITIONS
            if fhir_action.condition:
                generated_structural_elements[llm][pname]["conditions"][fhir_action.id] = fhir_action.condition.label
        
        ## RELATIONSHIPS
        generated_structural_elements[llm][pname]["relationships"] = [rel for rel in output.relationships]


### Action Matching

Before analyze all structural output components, we need to match generated actions with the correspondent ground truth action (if exists)

In [91]:
from rapidfuzz import fuzz
import pandas as pd

actions_temp = {}

for llm, plan_outputs in generated_structural_elements.items():
    actions_temp[llm] = {}
    for pname, output in plan_outputs.items():
        matching = {'NO_GEN' : []}
        for gt_id, gt_action in groundtruth_elements[pname].actions.items():
            max_sim = 0
            max_gen_id = None
            for gen_id, gen_action in output['actions'].items():
                sim = fuzz.token_set_ratio(gen_action, gt_action.label)
                if sim > max_sim:
                    max_sim = sim
                    max_gen_id = gen_id
            matching[gt_id] = max_gen_id
        actions_temp[llm][pname] = matching
        for gt_id, gt_action in groundtruth_elements[pname].actions.items():
            if gt_id not in actions_temp[llm][pname].keys():
                actions_temp[llm][pname]["NO_GEN"].append(f"{gt_id} - {gt_action.label}")

for llm, plan_outputs in generated_structural_elements.items():
    for pname, output in plan_outputs.items():
        no_match = []
        for gen_id, gen_action in output['actions'].items():
            if gen_id not in actions_temp[llm][pname].values():
                no_match.append(f"{gen_id} - {gen_action}")
        actions_temp[llm][pname]['NO_GT'] = '\n'.join(no_match)

from src.sources.llm import llm_models

action_matching_temp = []
for llm in llm_models.keys():
    for pname in plan_names:
        gt_acts = {gt_id: gt_action.label for gt_id, gt_action in groundtruth_elements[pname].actions.items()}
        gen_acts = {gen_id: gen_action for gen_id, gen_action in generated_structural_elements[llm][pname]['actions'].items()}
        for gt_id in gt_acts.keys():
            gen_id = actions_temp[llm][pname][gt_id]
            gen_lab = gen_acts.get(gen_id) if gen_id is not None else None
            action_matching_temp.append({
                'llm' : llm,
                'plan_id': pname,
                'gt_action_id': gt_id,
                'gt_action_label': gt_acts[gt_id],
                'gen_action_id': gen_id,
                'gen_action_label': gen_lab,
                'sim': fuzz.token_set_ratio(gen_lab, gt_acts[gt_id])
            })
        action_matching_temp.append({
            'llm' : llm,
            'plan_id': pname,
            'gt_action_id': None,
            'gt_action_label': None,
            'gen_action_id': "GEN_NO_MATCH",
            'gen_action_label': actions_temp[llm][pname]['NO_GT'],
            'sim': None
        })
        action_matching_temp.append({
            'llm' : llm,
            'plan_id': pname,
            'gt_action_id': "GEN_NO_MATCH",
            'gt_action_label': actions_temp[llm][pname]['NO_GEN'],
            'gen_action_id': None,
            'gen_action_label': None,
            'sim': None
        })
pd.DataFrame(action_matching_temp).to_csv("evaluation/rq1/helpers/action_matching_temp.csv", index=False)

After the extraction process, a manual check is needed and then reload action matching table

In [92]:
action_matches_df = pd.read_csv('evaluation/rq1/helpers/actions_matching.csv', sep=";")
action_matches_df.head(10)

,llm,plan_id,gt_action_id,gt_action_label,gen_action_id,gen_action_label
0,llama,P01,A5,The patient then performs the prescribed exerc...,A5,the patient performs the rehabilitation exerci...
1,llama,P01,A4,The patient receives education on generic home...,A4,receives education on generic home rehabilitat...
2,llama,P01,A3,the patient completes a clinical questionnaire,A3,the patient completes a clinical questionnaire
3,llama,P01,A2,"One week later, a specialist follow-up visit i...",A2,a specialist follow-up visit is performed
4,llama,P01,A1,The patient undergoes surgery procedure,A1,The patient undergoes surgery procedure.
5,llama,P01,NaN,NaN,GEN_NO_MATCH,NaN
6,llama,P02,A5,The patient receives post-stroke lifestyle edu...,A5,subsequently receives post-stroke lifestyle ed...
7,llama,P02,A4,the patient attends attends a cardiology speci...,A4,the patient attends a cardiology specialist fo...
8,llama,P02,A3,the practitioner is alerted with urgent priori...,A3,the practitioner is alerted with urgent priori...
9,llama,P02,A2,vital signs are monitored for ten days,A2,vital signs are monitored for ten days


In [93]:
gt_gen_action_matches = {}
gen_gt_action_matches = {}
gen_no_matches = {}

for llm, rows in action_matches_df.groupby('llm'):
    gt_gen_action_matches[llm] = {}
    gen_gt_action_matches[llm] = {}
    gen_no_matches[llm] = {}
    for plan_id, act_match in rows.groupby('plan_id'):
        gt_gen_action_matches[llm][plan_id] = {}
        gen_gt_action_matches[llm][plan_id] = {}
        gen_no_matches[llm][plan_id] = {}
        for _, row in act_match.iterrows():
            if row.gen_action_id == 'GEN_NO_MATCH':
                if not pd.isna(row.gen_action_label):
                    for miss in row.gen_action_label.split('\n'):
                        gen_no_matches[llm][plan_id][miss.split(' - ')[0]] = miss.split(' - ')[-1]
            else:
                gt_gen_action_matches[llm][plan_id][row.gt_action_id] = row.gen_action_id if not pd.isna(row.gen_action_id) else None
                if not pd.isna(row.gen_action_id):
                    gen_gt_action_matches[llm][plan_id][row.gen_action_id] = row.gt_action_id

### Action Analysis

In [94]:
# ACTIONS
action_analysis = {}
tp_actions_gt_key = {} # gt_id : gen_id
tp_actions_gen_key = {} # gen_id : gt_id

for llm in llm_models:
    action_analysis[llm] = []
    tp_actions_gt_key[llm] = {}
    tp_actions_gen_key[llm] = {}
    for plan_id in plan_names:
        tp_actions_gt_key[llm][plan_id] = {}
        tp_actions_gen_key[llm][plan_id] = {}
        action_analysis[llm].append({
            'plan_id' : plan_id,
            'tp' : len([gt_id for gt_id, gen_id in gt_gen_action_matches[llm][plan_id].items() if gen_id is not None]),
            'fp' : len(gen_no_matches[llm][plan_id]),
            'fn' : len([gt_id for gt_id, gen_id in gt_gen_action_matches[llm][plan_id].items() if gen_id is None])
        })
        for gt_id, gen_id in gt_gen_action_matches[llm][plan_id].items():
            if gen_id is not None:
                tp_actions_gt_key[llm][plan_id][gt_id] = gen_id
                tp_actions_gen_key[llm][plan_id][gen_id] = gt_id

action_analysis_summary = []
for llm in llm_models:
    tp_total = sum([plan['tp'] for plan in action_analysis[llm]])
    fp_total = sum([plan['fp'] for plan in action_analysis[llm]])
    fn_total = sum([plan['fn'] for plan in action_analysis[llm]])
    prec = compute_precision(tp_total, fp_total)
    rec = compute_recall(tp_total, fn_total)
    action_analysis_summary.append({
        'llm': llm,
        'tp' : tp_total,
        'fp' : fp_total,
        'fn' : fn_total,
        'precision': prec,
        'recall': rec,
        'f1': compute_f1_score(prec, rec)
    })

action_analysis_summary_df = pd.DataFrame(action_analysis_summary)
action_analysis_summary_df.to_csv("evaluation/rq1/action_analysis_summary.csv")
action_analysis_summary_df

,llm,tp,fp,fn,precision,recall,f1
0,mistral,88,7,1,0.926316,0.988764,0.956522
1,mixtral_8x7,87,4,2,0.956044,0.977528,0.966667
2,mixtral_8x22,82,1,7,0.987952,0.921348,0.953488
3,llama,85,0,4,1.000000,0.955056,0.977011
4,phi4,85,1,4,0.988372,0.955056,0.971429


### Relationships Analysis

In [95]:
relationships_gt = {}

for plan_id in plan_names:
    relationships_gt[plan_id] = []
    for rel in groundtruth_elements[plan_id].relationships:
        relationships_gt[plan_id].append((rel.from_action, rel.to_action))

relationships_gen = {}

for llm in llm_models:
    relationships_gen[llm] = {}
    for plan_id in plan_names:
        relationships_gen[llm][plan_id] = []
        for gen_rel in generated_structural_elements[llm][plan_id]["relationships"]:
            # if gen_rel.from_action and gen_rel.to_action:
            relationships_gen[llm][plan_id].append((
                gen_gt_action_matches[llm][plan_id].get(gen_rel.from_action),
                gen_gt_action_matches[llm][plan_id].get(gen_rel.to_action)
            ))

In [96]:
# Relationships
relationships_analysis = {}

for llm in llm_models:
    relationships_analysis[llm] = []
    for plan_id in plan_names:
        tp = 0
        fp = 0
        fn = 0
        for gen_rel in relationships_gen[llm][plan_id]:
            if gen_rel in relationships_gt[plan_id]:
                tp += 1
            else:
                fp += 1
        for gt_rel in relationships_gt[plan_id]:
            if gt_rel not in relationships_gen[llm][plan_id]:
                fn += 1
        relationships_analysis[llm].append({
            'plan_id': plan_id,
            'tp': tp,
            'fp': fp,
            'fn': fn,
        })

relationships_analysis_summary = []
for llm in llm_models:
    tp_total = sum([plan['tp'] for plan in relationships_analysis[llm]])
    fp_total = sum([plan['fp'] for plan in relationships_analysis[llm]])
    fn_total = sum([plan['fn'] for plan in relationships_analysis[llm]])
    prec = compute_precision(tp_total, fp_total)
    rec = compute_recall(tp_total, fn_total)
    relationships_analysis_summary.append({
        'llm': llm,
        'tp' : tp_total,
        'fp' : fp_total,
        'fn' : fn_total,
        'precision': prec,
        'recall': rec,
        'f1': compute_f1_score(prec, rec)
    })

relationships_analysis_summary_df = pd.DataFrame(relationships_analysis_summary)
relationships_analysis_summary_df.to_csv("evaluation/rq1/relationships_analysis_summary.csv")
relationships_analysis_summary_df

,llm,tp,fp,fn,precision,recall,f1
0,mistral,34,28,31,0.548387,0.523077,0.535433
1,mixtral_8x7,51,22,14,0.698630,0.784615,0.739130
2,mixtral_8x22,49,12,16,0.803279,0.753846,0.777778
3,llama,51,12,14,0.809524,0.784615,0.796875
4,phi4,49,14,16,0.777778,0.753846,0.765625


### Temporal Constraints Analysis

In [97]:
## Temporal Constraints
temporal_constraints_gt = {}

for plan_id in plan_names:
    temporal_constraints_gt[plan_id] = {}
    for action_id, action in groundtruth_elements[plan_id].actions.items():
        temporal_constraints_gt[plan_id][action.id] = action.temporal_constraint

temporal_constraints_gen = {}
for llm in llm_models:
    temporal_constraints_gen[llm] = {}
    for plan_id in plan_names:
        temporal_constraints_gen[llm][plan_id] = {}
        for action in generated_outputs[llm][plan_id].fhir_actions:
            temporal_constraints_gen[llm][plan_id][action.id] = action.fhir_temporal_constraint

In [98]:
temporal_constraints_label_and_type = []
for llm in llm_models:
    for plan_id in plan_names:
        for action in generated_outputs[llm][plan_id].fhir_actions:
            if action.temporal_constraint:
                gt_id = gen_gt_action_matches[llm][plan_id].get(action.id)
                if gt_id:
                    gt_tc = temporal_constraints_gt[plan_id].get(gt_id)
                else:
                    gt_tc = None
                temporal_constraints_label_and_type.append({
                    "llm" : llm,
                    "pid" : plan_id,
                    "gen_id" : action.id,
                    "gen_tc_label" : action.temporal_constraint.label,
                    "gen_tc_type": action.temporal_constraint.type.value,
                    "gt_id": gt_id,
                    "gt_tc_label" : gt_tc.label if gt_tc else None,
                    "gt_tc_type": gt_tc.type.value if gt_tc else None,
                })
pd.DataFrame(temporal_constraints_label_and_type).to_csv("evaluation/rq1/helpers/temporal_constraints_label_and_type.csv", index=False)

In [99]:
# TEMPORAL CONSTRAINTS
tc_analysis = {}
tc_detailed_analysis = {}

for llm in llm_models:
    tc_analysis[llm] = []

for llm in llm_models:
    tc_detailed_analysis[llm] = {}
    for pid in plan_names:
        tc_detailed_analysis[llm][pid] = {
            'tp': [],
            'fp': [],
            'fn': []
        }
        tp = 0
        fp_sp = 0
        fp_wtc = 0
        fp_no_match = 0
        fn_sp = 0
        fn_no_match = 0
        for gt_id, gt_action in groundtruth_elements[pid].actions.items():
            gen_tc = None
            gen_id = gt_gen_action_matches[llm][pid][gt_id]
            if gen_id is not None:
                # MATCH
                for gen_action in generated_outputs[llm][pid].fhir_actions:
                    if gen_action.id == gen_id:
                        gen_tc = gen_action.temporal_constraint
                if gen_tc is None:
                    # TC not in GEN
                    if gt_action.temporal_constraint is not None:
                        fn_sp += 1
                        tc_detailed_analysis[llm][pid]['fn'].append(gt_id)
                else:
                    # TC in GEN
                    if gt_action.temporal_constraint is None:
                        fp_sp += 1
                        tc_detailed_analysis[llm][pid]['fp'].append(gt_id)
                    else:
                        # TC in GT
                        if gt_action.temporal_constraint.type != gen_tc.type:
                            fp_wtc += 1
                            tc_detailed_analysis[llm][pid]['fp'].append(gt_id)
                        else:
                            tp += 1
                            tc_detailed_analysis[llm][pid]['tp'].append(gt_id)
            else:
                # NO MATCH (GT yes, GEN no)
                if gt_action.temporal_constraint is not None:
                    fn_no_match += 1
        for gen_act_id in gen_no_matches[llm][pid].keys():
            for gen_action in generated_outputs[llm][pid].fhir_actions:
                if gen_action.id == gen_act_id and gen_action.temporal_constraint is not None:
                    fp_no_match += 1
        tc_analysis[llm].append({
            'plan_id': pid,
            'tp' : tp,
            'fp_sp' : fp_sp,
            'fp_wtc' : fp_wtc,
            'fp_no_match' : fp_no_match,
            'fn_sp' : fn_sp,
            'fn_no_match' : fn_no_match,
        })

In [100]:
tc_analysis_summary = {}
for llm in llm_models:
    tp_total = sum([plan['tp'] for plan in tc_analysis[llm]])
    fp_sp_total = sum([plan['fp_sp'] for plan in tc_analysis[llm]])
    fp_wtc_total = sum([plan['fp_wtc'] for plan in tc_analysis[llm]])
    fp_no_match_total = sum([plan['fp_no_match'] for plan in tc_analysis[llm]])
    fn_sp_total = sum([plan['fn_sp'] for plan in tc_analysis[llm]])
    fn_no_match_total = sum([plan['fn_no_match'] for plan in tc_analysis[llm]])
    fp_total = fp_sp_total + fp_wtc_total + fp_no_match_total
    fn_total = fn_sp_total + fn_no_match_total
    prec = compute_precision(tp_total, fp_total)
    rec = compute_recall(tp_total, fn_total)
    tc_analysis_summary[llm] = {
        'tp' : tp_total,
        'fp' : fp_total,
        'fn' : fn_total,
        'precision': prec,
        'recall': rec,
        'f1': compute_f1_score(prec, rec),
        'fp_sp_total' : fp_sp_total,
        'fp_wtc_total' : fp_wtc_total,
        'fp_no_match_total' : fp_no_match_total,
        'fn_sp_total' : fn_sp_total,
        'fn_no_match_total' : fn_no_match_total,
    }
tc_analysis_summary_df = pd.DataFrame([
    {"llm": llm, **metrics}
    for llm, metrics in tc_analysis_summary.items()
])
tc_analysis_summary_df.to_csv('evaluation/rq1/tc_analysis_summary.csv', index=False)
tc_analysis_summary_df

,llm,tp,fp,fn,precision,recall,f1,fp_sp_total,fp_wtc_total,fp_no_match_total,fn_sp_total,fn_no_match_total
0,mistral,4,0,22,1.000000,0.153846,0.266667,0,0,0,22,0
1,mixtral_8x7,5,2,20,0.714286,0.200000,0.312500,0,1,1,19,1
2,mixtral_8x22,20,6,5,0.769231,0.800000,0.784314,5,1,0,4,1
3,llama,23,5,2,0.821429,0.920000,0.867925,4,1,0,1,1
4,phi4,25,1,1,0.961538,0.961538,0.961538,1,0,0,1,0


In [102]:
tot_tc = 0
for pid in plan_names:
    for _, gt_action in groundtruth_elements[pid].actions.items():
        if gt_action.temporal_constraint:
            tot_tc += 1

for llm in llm_models:
    tc_in_match = 0
    for plan_id in plan_names:
        for gt_id, gt_action in groundtruth_elements[plan_id].actions.items():
            if gt_gen_action_matches[llm][plan_id][gt_id] is not None and gt_action.temporal_constraint is not None:
                tc_in_match += 1
    print(llm, 'tc:', tc_in_match, "/", tot_tc)

mistral tc: 26 / 26
mixtral_8x7 tc: 25 / 26
mixtral_8x22 tc: 25 / 26
llama tc: 25 / 26
phi4 tc: 26 / 26


### Condition Analysis

In [103]:
# CONDITIONS
conditions_gt = {}

for plan_id in plan_names:
    conditions_gt[plan_id] = {}
    for action_id, action in groundtruth_elements[plan_id].actions.items():
        conditions_gt[plan_id][action.id] = action.condition.label if action.condition else None

conditions_gen = {}
for llm in llm_models:
    conditions_gen[llm] = {}
    for plan_id in plan_names:
        conditions_gen[llm][plan_id] = {}
        for action in generated_outputs[llm][plan_id].fhir_actions:
            conditions_gen[llm][plan_id][action.id] = action.condition.label if action.condition else None

In [104]:
conditions_label = []
for llm in llm_models:
    for plan_id in plan_names:
        for action in generated_outputs[llm][plan_id].fhir_actions:
            if action.condition:
                gt_id = gen_gt_action_matches[llm][plan_id].get(action.id)
                if gt_id:
                    gt_cond = conditions_gt[plan_id].get(gt_id)
                else:
                    gt_cond = None
                conditions_label.append({
                    "llm" : llm,
                    "pid" : plan_id,
                    "gen_id" : action.id,
                    "gen_cond_label" : action.condition.label,
                    "gt_id": gt_id,
                    "gt_cond_label" : gt_cond,
                    "sim": fuzz.token_set_ratio(action.condition.label, gt_cond) if gt_cond else None
                })
pd.DataFrame(conditions_label).to_csv("evaluation/rq1/helpers/conditions_label.csv", index=False)

In [105]:
conditions_analysis = {}
cond_detailed_analysis = {}

for llm in llm_models:
    conditions_analysis[llm] = []

for llm in llm_models:
    cond_detailed_analysis[llm] = {}
    for pid in plan_names:
        cond_detailed_analysis[llm][pid] = {
            'tp': [],
            'fp': [],
            'fn': []
        }
        tp = 0
        fp_sp = 0
        fp_no_match = 0
        fn_sp = 0
        fn_no_match = 0
        for gt_id, gt_action in groundtruth_elements[pid].actions.items():
            gen_tc = None
            gen_id = gt_gen_action_matches[llm][pid][gt_id]
            if gen_id is not None:
                # MATCH
                for gen_action in generated_outputs[llm][pid].fhir_actions:
                    if gen_action.id == gen_id:
                        gen_tc = gen_action.condition
                if gen_tc is None:
                    # Condition not in GEN
                    if gt_action.condition is not None:
                        fn_sp += 1
                        cond_detailed_analysis[llm][pid]['fn'].append(gt_id)
                else:
                    # Condition in GEN
                    if gt_action.condition is None:
                        fp_sp += 1
                        cond_detailed_analysis[llm][pid]['fp'].append(gt_id)
                    else:
                        # Condition in GT
                        tp += 1
                        cond_detailed_analysis[llm][pid]['tp'].append(gt_id)
            else:
                # NO MATCH (GT yes, GEN no)
                if gt_action.condition is not None:
                    fn_no_match += 1
        for gen_act_id in gen_no_matches[llm][pid].keys():
            for gen_action in generated_outputs[llm][pid].fhir_actions:
                if gen_action.id == gen_act_id and gen_action.condition is not None:
                    fp_no_match += 1
        conditions_analysis[llm].append({
            'plan_id': pid,
            'tp' : tp,
            'fp_sp' : fp_sp,
            'fp_no_match' : fp_no_match,
            'fn_sp' : fn_sp,
            'fn_no_match' : fn_no_match,
        })

In [106]:
conditions_analysis_summary = {}
for llm in llm_models:
    tp_total = sum([plan['tp'] for plan in conditions_analysis[llm]])
    fp_sp_total = sum([plan['fp_sp'] for plan in conditions_analysis[llm]])
    fp_no_match_total = sum([plan['fp_no_match'] for plan in conditions_analysis[llm]])
    fn_sp_total = sum([plan['fn_sp'] for plan in conditions_analysis[llm]])
    fn_no_match_total = sum([plan['fn_no_match'] for plan in conditions_analysis[llm]])
    fp_total = fp_sp_total + fp_no_match_total
    fn_total = fn_sp_total + fn_no_match_total
    prec = compute_precision(tp_total, fp_total)
    rec = compute_recall(tp_total, fn_total)
    conditions_analysis_summary[llm] = {
        'tp' : tp_total,
        'fp' : fp_total,
        'fn' : fn_total,
        'precision': prec,
        'recall': rec,
        'f1': compute_f1_score(prec, rec),
        'fp_sp_total' : fp_sp_total,
        'fp_no_match_total' : fp_no_match_total,
        'fn_sp_total' : fn_sp_total,
        'fn_no_match_total' : fn_no_match_total,
    }
conditions_analysis_summary_df = pd.DataFrame([
    {"llm": llm, **metrics}
    for llm, metrics in conditions_analysis_summary.items()
])
conditions_analysis_summary_df.to_csv('evaluation/rq1/conditions_analysis_summary.csv', index=False)
conditions_analysis_summary_df

,llm,tp,fp,fn,precision,recall,f1,fp_sp_total,fp_no_match_total,fn_sp_total,fn_no_match_total
0,mistral,3,0,17,1.000000,0.15,0.260870,0,0,17,0
1,mixtral_8x7,10,1,10,0.909091,0.50,0.645161,1,0,10,0
2,mixtral_8x22,13,6,7,0.684211,0.65,0.666667,5,1,5,2
3,llama,18,0,2,1.000000,0.90,0.947368,0,0,2,0
4,phi4,20,3,0,0.869565,1.00,0.930233,3,0,0,0


In [128]:
tot_cond = 0
for pid in plan_names:
    for _, gt_action in groundtruth_elements[pid].actions.items():
        if gt_action.condition:
            tot_cond += 1

for llm in llm_models:
    cond_in_match = 0
    for plan_id in plan_names:
        for gt_id, gt_action in groundtruth_elements[plan_id].actions.items():
            if gt_gen_action_matches[llm][plan_id][gt_id] is not None and gt_action.condition is not None:
                cond_in_match += 1
    print(llm, 'cond:', cond_in_match, "/", tot_cond)

mistral cond: 20 / 20
mixtral_8x7 cond: 20 / 20
mixtral_8x22 cond: 18 / 20
llama cond: 20 / 20
phi4 cond: 20 / 20


## RQ2 - How accurately does SHAPE semantically ground the extracted clinical knowledge?

This research question evaluates the semantic grounding capabilities of the proposed methodology, including
ActivityDefinition selection during Clinical Activity Grounding and the resolution of clinical parameters. It also
assesses terminology grounding to SNOMED CT concepts and FHIR ValueSets during the FHIR Representation
Generation stage.

In [108]:
import os

os.makedirs("evaluation/rq2", exist_ok=True)

### ActivityDefinition Grounding Analysis

In [109]:
activity_grounding_groundtruth = {}
for pid, gt_plan in groundtruth_elements.items():
    activity_grounding_groundtruth[pid] = {}
    for gt_action_id, gt_action in gt_plan.actions.items():
        activity_grounding_groundtruth[pid][gt_action_id] = gt_action.activity_definition.split("/")[-1]

activity_grounding_generated = {}
activity_grounding_infos_generated = {}
for llm in llm_models:
    activity_grounding_generated[llm] = {}
    activity_grounding_infos_generated[llm] = {}
    for pid in plan_names:
        activity_grounding_generated[llm][pid] = {}
        activity_grounding_infos_generated[llm][pid] = {}
        for gen_action in generated_outputs[llm][pid].fhir_actions:
            activity_grounding_generated[llm][pid][gen_action.id] = gen_action.activity_definition.id if gen_action.activity_definition else None
            activity_grounding_infos_generated[llm][pid][gen_action.id] = {
                "candidates" : gen_action.candidates,
                "confidence" : gen_action.confidence,
            }

In [ ]:
## Activity Grounding
ad_grounding_evaluation = {}
for llm in llm_models:
    ad_grounding_evaluation[llm] = []

for llm in llm_models:
    for plan_id in plan_names:
        
        n_actions = 0
        hits_k1 = 0
        hits_k2 = 0
        hits_k3 = 0
        rr_sum = 0
        correct = 0
        retrieval_error = 0
        selection_error = 0
        none_selection = 0

        for gt_act_id, gt_ad in activity_grounding_groundtruth[plan_id].items():
            gen_act_id = gt_gen_action_matches[llm][plan_id][gt_act_id]
            if gen_act_id is None:
                continue
            gen_candidates = activity_grounding_infos_generated[llm][plan_id][gen_act_id]['candidates']
            gen_ad = activity_grounding_generated[llm][plan_id][gen_act_id]
            if gen_ad is None:
                none_selection += 1

            n_actions += 1
            
            gt_rank = None
            for candidate in gen_candidates:
                rank = candidate.retrieval_rank
                is_gold = candidate.activity_definition.id == activity_grounding_groundtruth[plan_id][gt_act_id]
                if is_gold:
                    gt_rank = rank
            if gt_rank is not None:
                if gt_rank <= 1:
                    hits_k1 += 1
                if gt_rank <= 2:
                    hits_k2 += 1
                if gt_rank <= 3:
                    hits_k3 += 1
                rr_sum += 1.0 / gt_rank

                if gen_ad == activity_grounding_groundtruth[plan_id][gt_act_id]:
                    correct += 1
                else:
                    selection_error += 1
            else:
                retrieval_error += 1
        
        recall_at_1 = hits_k1 / n_actions if n_actions else 0
        recall_at_2 = hits_k2 / n_actions if n_actions else 0
        recall_at_3 = hits_k3 / n_actions if n_actions else 0
        mrr = rr_sum / n_actions if n_actions else 0
            
        selection_accuracy = correct / hits_k3 if hits_k3 else 0
        e2e_accuracy = correct / n_actions if n_actions else 0
        retrieval_error_rate = retrieval_error / n_actions if n_actions else 0
        selection_error_rate = selection_error / n_actions if n_actions else 0
            
        ad_grounding_evaluation[llm].append({
            "pathway_id": plan_id,
            "recall_at_1": recall_at_1,
            "recall_at_2": recall_at_2,
            "recall_at_3": recall_at_3,
            "mrr": mrr,
            'selection_accuracy' : selection_accuracy,
            'e2e_accuracy' : e2e_accuracy,
            "retrieval_error_rate" : retrieval_error_rate,
            "selection_error_rate" : selection_error_rate,
            "n_actions": n_actions,
            "correct": correct,
            "hits_k1": hits_k1,
            "hits_k2": hits_k2,
            "hits_k3": hits_k3,
            "rr_sum": rr_sum,
            "retrieval_errors" : retrieval_error,
            "selection_errors" : selection_error,
            "none_selection" : none_selection
        })

In [111]:
ad_grounding_evaluation_summary = {}
for llm in llm_models:
    n_actions = sum([stat['n_actions'] for stat in ad_grounding_evaluation[llm]])
    hits_k1 = sum([stat['hits_k1'] for stat in ad_grounding_evaluation[llm]])
    hits_k2 = sum([stat['hits_k2'] for stat in ad_grounding_evaluation[llm]])
    hits_k3 = sum([stat['hits_k3'] for stat in ad_grounding_evaluation[llm]])
    recall_at_1 = hits_k1 / n_actions
    recall_at_2 = hits_k2 / n_actions
    recall_at_3 = hits_k3 / n_actions
    mrr = sum([stat['rr_sum'] for stat in ad_grounding_evaluation[llm]]) / n_actions
        
    selection_accuracy = sum([stat['correct'] for stat in ad_grounding_evaluation[llm]]) / hits_k3 if hits_k3 else 0
    e2e_accuracy = sum([stat['correct'] for stat in ad_grounding_evaluation[llm]]) / n_actions
    retrieval_error_rate = sum([stat['retrieval_errors'] for stat in ad_grounding_evaluation[llm]]) / n_actions
    selection_error_rate = sum([stat['selection_errors'] for stat in ad_grounding_evaluation[llm]]) / n_actions
    ad_grounding_evaluation_summary[llm] = {
        "recall_at_1": recall_at_1,
        "recall_at_2": recall_at_2,
        "recall_at_3": recall_at_3,
        "mrr": mrr,
        'selection_accuracy' : selection_accuracy,
        'e2e_accuracy' : e2e_accuracy,
        "retrieval_error_rate" : retrieval_error_rate,
        "selection_error_rate" : selection_error_rate,
        "n_actions" : n_actions,
        "hits_k1" : hits_k1,
        "hits_k2" : hits_k2,
        "hits_k3" : hits_k3,
        "correct": sum([stat['correct'] for stat in ad_grounding_evaluation[llm]]),
        "retrieval_errors" : sum([stat['retrieval_errors'] for stat in ad_grounding_evaluation[llm]]),
        "selection_errors" : sum([stat['selection_errors'] for stat in ad_grounding_evaluation[llm]]),
        "none_selection" : sum([stat['none_selection'] for stat in ad_grounding_evaluation[llm]]),
    }
ad_grounding_evaluation_summary_df= pd.DataFrame([
    {"llm": llm, **metrics}
    for llm, metrics in ad_grounding_evaluation_summary.items()
])
ad_grounding_evaluation_summary_df.to_csv('evaluation/rq2/ad_grounding_evaluation_summary.csv', index=False)
ad_grounding_evaluation_summary_df

,llm,recall_at_1,recall_at_2,recall_at_3,mrr,selection_accuracy,e2e_accuracy,retrieval_error_rate,selection_error_rate,n_actions,hits_k1,hits_k2,hits_k3,correct,retrieval_errors,selection_errors,none_selection
0,mistral,0.795455,0.829545,0.875000,0.827652,0.974026,0.852273,0.125000,0.022727,88,70,73,77,75,11,2,0
1,mixtral_8x7,0.770115,0.839080,0.862069,0.812261,0.893333,0.770115,0.137931,0.091954,87,67,73,75,67,12,8,13
2,mixtral_8x22,0.768293,0.841463,0.853659,0.808943,0.985714,0.841463,0.146341,0.012195,82,63,69,70,69,12,1,1
3,llama,0.776471,0.847059,0.882353,0.823529,0.986667,0.870588,0.117647,0.011765,85,66,72,75,74,10,1,4
4,phi4,0.776471,0.835294,0.870588,0.817647,0.945946,0.823529,0.129412,0.047059,85,66,71,74,70,11,4,7


In [112]:
# ONLY Exact Matching AD Grounding for TP Actions

tp_ad_grounding_gt_key = {} # gt_id : AD ID
tp_ad_grounding_gen_key = {} # gen_id : AD ID

for llm in llm_models:
    tp_ad_grounding_gt_key[llm] = {}
    tp_ad_grounding_gen_key[llm] = {}
    for plan_id in plan_names:
        tp_ad_grounding_gt_key[llm][plan_id] = {}
        tp_ad_grounding_gen_key[llm][plan_id] = {}
        for gt_action_id, gen_action_id in tp_actions_gt_key[llm][plan_id].items():
            if activity_grounding_generated[llm][plan_id][gen_action_id] == activity_grounding_groundtruth[plan_id][gt_action_id]:
                tp_ad_grounding_gt_key[llm][plan_id][gt_action_id] = activity_grounding_groundtruth[plan_id][gt_action_id]
                tp_ad_grounding_gen_key[llm][plan_id][gen_action_id] = activity_grounding_groundtruth[plan_id][gt_action_id]

### Parameter Identification Analysis

In [114]:
parameters_groundtruth = {}

for pid, gt_plan in groundtruth_elements.items():
    parameters_groundtruth[pid] = {}
    for gt_action_id, gt_action in gt_plan.actions.items():
        if gt_action.parameters:
            parameters_groundtruth[pid][gt_action_id] = gt_action.parameters
        else:
            parameters_groundtruth[pid][gt_action_id] = {}

generated_parameters = {}
for llm in llm_models:
    generated_parameters[llm] = {}
    for plan_id in plan_names:
        generated_parameters[llm][plan_id] = {}
        for gen_action in generated_outputs[llm][plan_id].fhir_actions:
            generated_parameters[llm][plan_id][gen_action.id] = {}
            for gen_param in gen_action.parameters:
                generated_parameters[llm][plan_id][gen_action.id][gen_param.path] = gen_param

for llm in llm_models:
    tot_param_count = 0
    possible_param_count = 0
    for plan_id in plan_names:
        for act_id in tp_ad_grounding_gt_key[llm][plan_id].keys():
            possible_param_count += len(parameters_groundtruth[plan_id][act_id])
        for act_id, params in parameters_groundtruth[plan_id].items():
            tot_param_count += len(params)
    print(llm, possible_param_count, "/", tot_param_count)

mistral 63 / 77
mixtral_8x7 60 / 77
mixtral_8x22 60 / 77
llama 61 / 77
phi4 59 / 77


In [115]:
# Parameter identification
tp_parameters_gt_key = {}
tp_parameters_gen_key = {}

param_identification_analysis = []

for llm in llm_models:
    tp = 0
    fp = 0
    fn = 0
    tp_parameters_gt_key[llm] = {}
    tp_parameters_gen_key[llm] = {}
    for plan_id in plan_names:
        tp_parameters_gt_key[llm][plan_id] = {}
        tp_parameters_gen_key[llm][plan_id] = {}
        for gt_act_id in tp_ad_grounding_gt_key[llm][plan_id].keys():
            gen_act_id = gt_gen_action_matches[llm][plan_id][gt_act_id]
            if gen_act_id is None:
                print(gen_act_id, "GEN NONE")
                continue
            tp_parameters_gt_key[llm][plan_id][gt_act_id] = []
            tp_parameters_gen_key[llm][plan_id][gen_act_id] = []
            for gen_path in generated_parameters[llm][plan_id][gen_act_id].keys():
                if gen_path in parameters_groundtruth[plan_id][gt_act_id].keys():
                    tp += 1
                    tp_parameters_gt_key[llm][plan_id][gt_act_id].append(gen_path)
                    tp_parameters_gen_key[llm][plan_id][gen_act_id].append(gen_path)
                else:
                    fp += 1
        for gen_act_id in tp_ad_grounding_gen_key[llm][plan_id].keys():
            gt_act_id = gen_gt_action_matches[llm][plan_id][gen_act_id]
            if gt_act_id is None:
                print(gt_act_id, "GT NONE")
                continue
            for gt_path in parameters_groundtruth[plan_id][gt_act_id].keys():
                if gt_path not in generated_parameters[llm][plan_id][gen_act_id].keys():
                    fn += 1
    prec = compute_precision(tp, fp)
    rec = compute_recall(tp, fn)
    param_identification_analysis.append({
        'llm': llm,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision' : prec,
        'recall' : rec,
        'f1-score' : compute_f1_score(prec, rec)
    })

param_identification_analysis_df= pd.DataFrame(param_identification_analysis)
param_identification_analysis_df.to_csv('evaluation/rq2/param_identification_analysis.csv', index=False)
param_identification_analysis_df

,llm,tp,fp,fn,precision,recall,f1-score
0,mistral,23,16,40,0.589744,0.365079,0.450980
1,mixtral_8x7,39,11,21,0.780000,0.650000,0.709091
2,mixtral_8x22,51,6,9,0.894737,0.850000,0.871795
3,llama,58,12,3,0.828571,0.950820,0.885496
4,phi4,40,6,19,0.869565,0.677966,0.761905


In [116]:
generated_fhir_parameters = {}
for llm in llm_models:
    generated_fhir_parameters[llm] = {}
    for plan_id in plan_names:
        generated_fhir_parameters[llm][plan_id] = {}
        for gen_action in generated_outputs[llm][plan_id].fhir_actions:
            generated_fhir_parameters[llm][plan_id][gen_action.id] = {}
            for gen_param in gen_action.fhir_parameters:
                generated_fhir_parameters[llm][plan_id][gen_action.id][gen_param.path] = gen_param

In [117]:
from src.model.enums import ParameterCategory

groundtruth_parameters_by_type = {}

for category in ParameterCategory:
    groundtruth_parameters_by_type[category] = {}
    for plan_id in plan_names:
        groundtruth_parameters_by_type[category][plan_id] = {}
        for gt_act_id, gt_params in parameters_groundtruth[plan_id].items():
            groundtruth_parameters_by_type[category][plan_id][gt_act_id] = {}
            for gt_path, gt_param in gt_params.items():
                if gt_param.category == category:
                    groundtruth_parameters_by_type[category][plan_id][gt_act_id][gt_path] = gt_param


generated_parameters_by_type = {}

for category in ParameterCategory:
    generated_parameters_by_type[category] = {}
    for llm in llm_models:
        generated_parameters_by_type[category][llm] = {}
        for plan_id in plan_names:
            generated_parameters_by_type[category][llm][plan_id] = {}
            for gen_act_id, gen_params in generated_fhir_parameters[llm][plan_id].items():
                generated_parameters_by_type[category][llm][plan_id][gen_act_id] = {}
                for gen_path, gen_param in gen_params.items():
                    if gen_param.category == category:
                        generated_parameters_by_type[category][llm][plan_id][gen_act_id][gen_path] = gen_param

In [118]:
param_tot_count = {cat : 0 for cat in ParameterCategory}
for plan_id in plan_names:
    for cat in ParameterCategory:
        for gt_act_id, gt_params in groundtruth_parameters_by_type[cat][plan_id].items():
            for gt_param in gt_params.values():
                param_tot_count[gt_param.category] += 1

param_count = {}
for llm in llm_models:
    param_count[llm] = {
        cat : 0 for cat in ParameterCategory
    }
    for plan_id in plan_names:
        for gen_act_id, gen_paths in tp_parameters_gen_key[llm][plan_id].items():
            for gen_path in gen_paths:
                gen_cat = generated_fhir_parameters[llm][plan_id][gen_act_id][gen_path].category
                param_count[llm][gen_cat] += 1
    
for llm in llm_models:
    for cat, count in param_count[llm].items():
        print(llm, cat.name, count, "/", param_tot_count[cat])
    print()

mistral ONTOLOGY 12 / 45
mistral REFERENCE 1 / 16
mistral VALUE_SET 7 / 9
mistral RESOURCE 0 / 4
mistral STRING 3 / 3

mixtral_8x7 ONTOLOGY 28 / 45
mixtral_8x7 REFERENCE 6 / 16
mixtral_8x7 VALUE_SET 5 / 9
mixtral_8x7 RESOURCE 0 / 4
mixtral_8x7 STRING 0 / 3

mixtral_8x22 ONTOLOGY 31 / 45
mixtral_8x22 REFERENCE 9 / 16
mixtral_8x22 VALUE_SET 8 / 9
mixtral_8x22 RESOURCE 0 / 4
mixtral_8x22 STRING 3 / 3

llama ONTOLOGY 31 / 45
llama REFERENCE 15 / 16
llama VALUE_SET 9 / 9
llama RESOURCE 0 / 4
llama STRING 3 / 3

phi4 ONTOLOGY 25 / 45
phi4 REFERENCE 4 / 16
phi4 VALUE_SET 9 / 9
phi4 RESOURCE 0 / 4
phi4 STRING 2 / 3



### FHIR ValueSet Grounding Analysis

In [119]:
## VALUESET GROUNDING

valueset_grounding_analysis = []

for llm in llm_models:
    tot_vs = param_count[llm][ParameterCategory.VALUE_SET]
    correct = 0
    not_correct = 0
    for plan_id in plan_names:
        for gen_act_id, paths in tp_parameters_gen_key[llm][plan_id].items():
            for path, gen_param in generated_parameters_by_type[ParameterCategory.VALUE_SET][llm][plan_id][gen_act_id].items():
                if path in paths:
                    gt_act_id = gen_gt_action_matches[llm][plan_id][gen_act_id]
                    gt_param = groundtruth_parameters_by_type[ParameterCategory.VALUE_SET][plan_id][gt_act_id][path]
                    if gt_param.value['value'] == gen_param.value['code']:
                        correct += 1
                    else:
                        not_correct += 1
    valueset_grounding_analysis.append({
        'llm': llm,
        'possible_paramaters_count' : tot_vs,
        'correct' : correct,
        'not_correct' : not_correct
    })
valueset_grounding_analysis_df = pd.DataFrame(valueset_grounding_analysis)
valueset_grounding_analysis_df.to_csv("evaluation/rq2/valueset_grounding_analysis.csv", index=False)
valueset_grounding_analysis_df

,llm,possible_paramaters_count,correct,not_correct
0,mistral,7,6,1
1,mixtral_8x7,5,5,0
2,mixtral_8x22,8,8,0
3,llama,9,9,0
4,phi4,9,9,0


### Ontology Grounding Analysis

In [121]:
from src.model.fhir_workflow import OntologyParameter

ontology_parameter_generated = {}
for llm in llm_models:
    ontology_parameter_generated[llm] = {}
    for plan_id in plan_names:
        ontology_parameter_generated[llm][plan_id] = {}
        with open(f"output/{llm}/{plan_id}_output.json", 'r') as f:
            gen_plan_output_dict = json.load(f)
        for gen_act_id, params in generated_parameters_by_type[ParameterCategory.ONTOLOGY][llm][plan_id].items():
            gen_action = None
            for action in gen_plan_output_dict['fhir_actions']:
                if action['id'] == gen_act_id:
                    gen_action = action
                    break
            ontology_parameter_generated[llm][plan_id][gen_act_id] = {}
            for path in params.keys():
                for gen_param in gen_action['fhir_parameters']:
                    if gen_param['path'] == path:
                        ontology_parameter_generated[llm][plan_id][gen_act_id][path] = OntologyParameter.from_dict(gen_param)

In [130]:
# ONTOLOGY GROUNDING

ontology_grounding_analysis = []

for llm in llm_models:
    n_params = param_count[llm][ParameterCategory.ONTOLOGY]
    hits_k1 = 0
    hits_k3 = 0
    hits_k5 = 0
    hits_k10 = 0
    rr_sum = 0
    correct = 0
    correct_none = 0
    retrieval_error = 0
    selection_error = 0
    
    for plan_id in plan_names:
        for gen_act_id, paths in tp_parameters_gen_key[llm][plan_id].items():
            for path, gen_param in ontology_parameter_generated[llm][plan_id][gen_act_id].items():
                if path in paths:
                    gt_act_id = gen_gt_action_matches[llm][plan_id][gen_act_id]
                    gt_param = groundtruth_parameters_by_type[ParameterCategory.ONTOLOGY][plan_id][gt_act_id][path]
                    
                    gt_codes = []
                    if gt_param.value:
                        gt_codes = [gt_param.value['coding'][0]['code']]
                    else:
                        gt_codes = [c['coding'][0]['code'] for c in gt_param.possible_values]
                    
                    gen_code = gen_param.value['coding'][0]['code'] if gen_param.value else None
                    gt_rank = None

                    for candidate in gen_param.candidates:
                        if candidate.ontology_concept.concept_id in gt_codes:
                            gt_rank = candidate.retrieval_rank
                            break
                    if gt_rank is not None:
                        if gt_rank <= 1:
                            hits_k1 += 1
                        if gt_rank <= 3:
                            hits_k3 += 1
                        if gt_rank <= 5:
                            hits_k5 += 1
                        if gt_rank <= 10:
                            hits_k10 += 1

                        rr_sum += 1.0 / gt_rank

                        if gen_code in gt_codes:
                            correct += 1
                        else:
                            if gen_code is None:
                                correct_none += 1
                            else:
                                selection_error += 1
                    else:
                        retrieval_error += 1

    
    recall_at_1 = hits_k1 / n_params if n_params else 0
    recall_at_3 = hits_k3 / n_params if n_params else 0
    recall_at_5 = hits_k5 / n_params if n_params else 0
    recall_at_10 = hits_k10 / n_params if n_params else 0
    mrr = rr_sum / n_params if n_params else 0
    selection_accuracy = correct / hits_k10 if hits_k10 else 0
    e2e_accuracy = correct / n_params if n_params else 0
    retrieval_error_rate = retrieval_error / n_params if n_params else 0
    selection_error_rate = selection_error / n_params if n_params else 0

    ontology_grounding_analysis.append({
        'llm': llm,
        "recall_at_1": recall_at_1,
        "recall_at_3": recall_at_3,
        "recall_at_5": recall_at_5,
        "recall_at_10": recall_at_10,
        "mrr": mrr,
        'selection_accuracy' : selection_accuracy,
        'e2e_accuracy' : e2e_accuracy,
        "retrieval_error_rate" : retrieval_error_rate,
        "selection_error_rate" : selection_error_rate,
        "n_params": n_params,
        "correct": correct,
        "hits_k1": hits_k1,
        "hits_k3": hits_k3,
        "hits_k5": hits_k5,
        "hits_k10": hits_k10,
        "rr_sum": rr_sum,
        "retrieval_errors" : retrieval_error,
        "selection_errors" : selection_error,
    })
ontology_grounding_analysis_df = pd.DataFrame(ontology_grounding_analysis)
ontology_grounding_analysis_df.to_csv("evaluation/rq2/ontology_grounding_analysis.csv", index=False)
ontology_grounding_analysis_df

,llm,recall_at_1,recall_at_3,recall_at_5,recall_at_10,mrr,selection_accuracy,e2e_accuracy,retrieval_error_rate,selection_error_rate,n_params,correct,hits_k1,hits_k3,hits_k5,hits_k10,rr_sum,retrieval_errors,selection_errors
0,mistral,0.083333,0.166667,0.416667,0.500000,0.183333,0.333333,0.166667,0.500000,0.333333,12,2,1,2,5,6,2.200000,6,4
1,mixtral_8x7,0.392857,0.428571,0.500000,0.571429,0.431888,0.312500,0.178571,0.428571,0.357143,28,5,11,12,14,16,12.092857,12,10
2,mixtral_8x22,0.419355,0.483871,0.548387,0.580645,0.466129,0.611111,0.354839,0.419355,0.225806,31,11,13,15,17,18,14.450000,13,7
3,llama,0.516129,0.580645,0.645161,0.677419,0.557527,0.809524,0.548387,0.322581,0.129032,31,17,16,18,20,21,17.283333,10,4
4,phi4,0.360000,0.440000,0.520000,0.600000,0.417048,0.466667,0.280000,0.400000,0.320000,25,7,9,11,13,15,10.426190,10,8


## RQ3 - Does SHAPE generate structurally valid and semantically correct FHIR PlanDefinition resources?

Does SHAPE generate structurally valid and semantically correct FHIR PlanDefinition resources?

This research question evaluates the overall quality of the generated FHIR PlanDefinition resources produced by
the complete SHAPE pipeline.

In [124]:
import os

os.makedirs('evaluation/rq3', exist_ok=True)

In [125]:
gt_actions_pd = {}
for plan_id in plan_names:
    gt_actions_pd[plan_id] = {}
    for action in plan_definitions_groundtruth[plan_id].action:
        gt_actions_pd[plan_id][action.linkId] = action

gen_actions_pd = {}
for llm in llm_models:
    gen_actions_pd[llm] = {}
    for plan_id in plan_names:
        gen_actions_pd[llm][plan_id] = {}
        with open(f"output/{llm}/{plan_id}_plandefinition.json", 'r') as f:
            gen_pd = PlanDefinition.model_validate(json.load(f))
        for action in gen_pd.action:
            gen_actions_pd[llm][plan_id][action.linkId] = action

### E2E PlanDefinition Analysis


In [ ]:
# E2E

all_actions = 89
all_relationships = 65
all_tc = 26
all_condition = 20
all_parameters = 77

e2e_analysis = {llm : {
    'llm': llm,
    'action_tp' : 0,
    'action_fp' : 0,
    'action_fn' : 0,
    'ad_tp' : 0,
    'ad_fp' : 0,
    'ad_fn' : 0,
    'rel_tp' : 0,
    'rel_fp' : 0,
    'rel_fn' : 0,
    'tc_find_tp' : 0,
    'tc_find_fp' : 0,
    'tc_find_fn' : 0,
    'tc_value_tp' : 0,
    'tc_value_fp' : 0,
    'tc_value_fn' : 0,
    'cond_tp' : 0,
    'cond_fp' : 0,
    'cond_fn' : 0,
    'param_find_tp' : 0,
    'param_find_fp' : 0,
    'param_find_fn' : 0,
    'param_value_tp' : 0,
    'param_value_fp' : 0,
    'param_value_fn' : 0,
} for llm in llm_models}

# From RQ1 Analysis:
# - action_tp	action_fp	action_fn
for info in action_analysis_summary:
    e2e_analysis[info['llm']]['action_tp'] = info['tp']
    e2e_analysis[info['llm']]['action_fp'] = info['fp']
    e2e_analysis[info['llm']]['action_fn'] = info['fn']

# - rel_tp	rel_fp	rel_fn
for info in relationships_analysis_summary:
    e2e_analysis[info['llm']]['rel_tp'] = info['tp']
    e2e_analysis[info['llm']]['rel_fp'] = info['fp']
    e2e_analysis[info['llm']]['rel_fn'] = info['fn']

# - cond_tp	cond_fp	cond_fn
for llm, info in conditions_analysis_summary.items():
    e2e_analysis[llm]['cond_tp'] = info['tp']
    e2e_analysis[llm]['cond_fp'] = info['fp']
    e2e_analysis[llm]['cond_fn'] = info['fn']

# ActivityDefinition:
for llm, info in ad_grounding_evaluation_summary.items():
    # - ad_tp (correct in RQ2)
    e2e_analysis[llm]['ad_tp'] = info['correct']
    # - ad_fp RQ2 (tot_action - none_selection - correct) + action_fp
    e2e_analysis[llm]['ad_fp'] = all_actions - info['correct'] - info["none_selection"] + e2e_analysis[llm]['action_fp']
    # - none_selection RQ2 + action_fn
    e2e_analysis[llm]['ad_fp'] =  info["none_selection"] + e2e_analysis[llm]['action_fn']

# TemporalConstraint - Duration & Repeat
for llm in llm_models:
    for plan_id in plan_names:
        for gt_act_id, gt_action in gt_actions_pd[plan_id].items():
            gen_id = gt_gen_action_matches[llm][plan_id][gt_act_id]
            if gt_action.timingTiming:
                if gen_id is None:
                    e2e_analysis[llm]['tc_find_fn'] += 1
                    e2e_analysis[llm]['tc_value_fn'] += 1
                else:
                    gen_action = gen_actions_pd[llm][plan_id][gen_id]
                    if gen_action.timingTiming:
                        e2e_analysis[llm]['tc_find_tp'] += 1
                        if gen_action.timingTiming == gt_action.timingTiming:
                            e2e_analysis[llm]['tc_value_tp'] += 1
                        else:
                            e2e_analysis[llm]['tc_value_fp'] += 1
                    else:
                        e2e_analysis[llm]['tc_find_fn'] += 1
                        e2e_analysis[llm]['tc_value_fn'] += 1
            else:
                if gen_id:
                    gen_action = gen_actions_pd[llm][plan_id][gen_id]
                    if gen_action.timingTiming:
                        e2e_analysis[llm]['tc_find_fp'] += 1
                        e2e_analysis[llm]['tc_value_fp'] += 1
        
        for gen_act_id, gen_action in gen_actions_pd[llm][plan_id].items():
            gt_id = gen_gt_action_matches[llm][plan_id].get(gen_act_id)
            if gt_id is None:
                if gen_action.timingTiming:
                    e2e_analysis[llm]['tc_find_fp'] += 1
                    e2e_analysis[llm]['tc_value_fp'] += 1

# TemporalConstraint - Offset
for llm in llm_models:
    for plan_id in plan_names:
        for gt_act_id, gt_action in gt_actions_pd[plan_id].items():
            gen_id = gt_gen_action_matches[llm][plan_id][gt_act_id]
            if gt_action.relatedAction and len(gt_action.relatedAction) > 0:
                gt_rel_act = gt_action.relatedAction[0]
                if gt_rel_act.offsetDuration:
                    if gen_id is None:
                        e2e_analysis[llm]['tc_find_fn'] += 1
                        e2e_analysis[llm]['tc_value_fn'] += 1
                    else:
                        gen_action = gen_actions_pd[llm][plan_id][gen_id]
                        if gen_action.relatedAction and len(gen_action.relatedAction) > 0:
                            gen_rel_act = gen_action.relatedAction[0]
                            if gen_rel_act.offsetDuration:
                                e2e_analysis[llm]['tc_find_tp'] += 1
                                if gt_rel_act.offsetDuration == gen_rel_act.offsetDuration:
                                    e2e_analysis[llm]['tc_value_tp'] += 1
                                else:
                                    e2e_analysis[llm]['tc_value_fp'] += 1
                            else:
                                e2e_analysis[llm]['tc_find_fn'] += 1
                                e2e_analysis[llm]['tc_value_fn'] += 1
                        else:
                            e2e_analysis[llm]['tc_find_fn'] += 1
                            e2e_analysis[llm]['tc_value_fn'] += 1
                else:
                    if gen_id:
                        gen_action = gen_actions_pd[llm][plan_id][gen_id]
                        if gen_action.relatedAction and len(gen_action.relatedAction) > 0:
                            gen_rel_act = gen_action.relatedAction[0]
                            if gen_rel_act.offsetDuration:
                                e2e_analysis[llm]['tc_find_fp'] += 1
                                e2e_analysis[llm]['tc_value_fp'] += 1

        for gen_act_id, gen_action in gen_actions_pd[llm][plan_id].items():
            gt_id = gen_gt_action_matches[llm][plan_id].get(gen_act_id)
            if gt_id is None:
                if gen_action.relatedAction and len(gen_action.relatedAction) > 0:
                    gen_rel_act = gen_action.relatedAction[0]
                    if gen_rel_act.offsetDuration:
                        e2e_analysis[llm]['tc_find_fp'] += 1
                        e2e_analysis[llm]['tc_value_fp'] += 1

# Parameters
for llm in llm_models:
    for plan_id in plan_names:
        for gt_act_id, gt_action in gt_actions_pd[plan_id].items():
            gen_id = gt_gen_action_matches[llm][plan_id][gt_act_id]
            for gt_dv in gt_action.dynamicValue or []:
                try:
                    gt_dv_value = json.loads(gt_dv.expression.expression)
                except:
                    gt_dv_value = gt_dv.expression.expression
                
                if gen_id is None:
                    e2e_analysis[llm]['param_find_fn'] += 1
                    e2e_analysis[llm]['param_value_fn'] += 1
                else:
                    gen_action = gen_actions_pd[llm][plan_id][gen_id]
                    gen_act_dynamic_values = {}
                    for dv in gen_action.dynamicValue or []:
                        gen_act_dynamic_values[dv.path] = dv.expression.expression
                    
                    gen_dv = gen_act_dynamic_values.get(gt_dv.path)
                    if gen_dv:
                        e2e_analysis[llm]['param_find_tp'] += 1
                        try:
                            gen_dv_value = json.loads(gen_dv)
                        except:
                            gen_dv_value = gen_dv
                        if gen_dv_value == gt_dv_value:
                            e2e_analysis[llm]['param_value_tp'] += 1
                        else:
                            e2e_analysis[llm]['param_value_fp'] += 1
                    else:
                        e2e_analysis[llm]['param_find_fn'] += 1
                        e2e_analysis[llm]['param_value_fn'] += 1
        
        for gen_act_id, gen_action in gen_actions_pd[llm][plan_id].items():
            gt_id = gen_gt_action_matches[llm][plan_id].get(gen_act_id)
            if gt_id is None:
                for gen_dv in gen_action.dynamicValue or []:
                    e2e_analysis[llm]['param_find_fp'] += 1
                    e2e_analysis[llm]['param_value_fp'] += 1

for llm, summary in e2e_analysis.items():
    summary["TP"] = summary['action_tp'] + summary['ad_tp'] + summary['rel_tp'] + summary['tc_find_tp'] + summary['tc_value_tp'] + summary['cond_tp'] + summary['param_find_tp'] + summary['param_value_tp']
    summary["FP"] = summary['action_fp'] + summary['ad_fp'] + summary['rel_fp'] + summary['tc_find_fp'] + summary['tc_value_fp'] + summary['cond_fp'] + summary['param_find_fp'] + summary['param_value_fp']
    summary["FN"] = summary['action_fn'] + summary['ad_fn'] + summary['rel_fn'] + summary['tc_find_fn'] + summary['tc_value_fn'] + summary['cond_fn'] + summary['param_find_fn'] + summary['param_value_fn']
    summary["precision"] = compute_precision(summary['TP'], summary['FP'])
    summary["recall"] = compute_recall(summary['TP'], summary['FN'])
    summary["F1-Score"] = compute_f1_score(summary["precision"], summary["recall"])

e2e_analysis_df = pd.DataFrame(list(e2e_analysis.values()))
e2e_analysis_df.to_csv('evaluation/rq3/e2e_analysis.csv', index=False)
e2e_analysis_df

,llm,action_tp,action_fp,action_fn,ad_tp,ad_fp,ad_fn,rel_tp,rel_fp,rel_fn,...,param_find_fn,param_value_tp,param_value_fp,param_value_fn,TP,FP,FN,precision,recall,F1-Score
0,mistral,88,7,1,75,1,0,34,28,31,...,51,4,28,51,230,70,203,0.766667,0.531178,0.627558
1,mixtral_8x7,87,4,2,67,15,0,51,22,14,...,35,2,40,35,263,88,140,0.749288,0.652605,0.697613
2,mixtral_8x22,82,1,7,69,8,0,49,12,16,...,23,4,50,23,295,105,86,0.737500,0.774278,0.755442
3,llama,85,0,4,74,8,0,51,12,14,...,16,12,49,16,342,82,58,0.806604,0.855000,0.830097
4,phi4,85,1,4,70,11,0,49,14,16,...,35,0,42,35,267,72,140,0.787611,0.656020,0.715818
